In [1]:
print('hi')

hi


In [2]:
import numpy as np

In [ ]:
# TODO - figure out how to replicate the splits used in the notebook for ALL folds

In [4]:
NPZ_PATH = "/data/adenocarcinoma_dataset.npz"
data = np.load(NPZ_PATH)

print(data.keys())
original_class_labels = data['labels1']
binarized_class_labels = data['labels2']
domain_text_list = data['domain_text']
sub_domain_text_list = data['sub_domain_text']

print("Unique labels1: ", np.unique(original_class_labels))
print("Unique labels2 (prediction target): ", np.unique(binarized_class_labels))
print("Unique domain_text: ", np.unique(domain_text_list))
print("Unique sub_domain_text: ", np.unique(sub_domain_text_list))

def get_mapping_from_paired_arrays(arr1: np.ndarray, arr2: np.ndarray) -> dict:
    """
    Given arrays arr1 and arr2, where an injective mapping exists from any i-th element in arr1 to the corresponding i-th element in arr2,
    return the mapping of unique values in arr1 to their corresponding value in arr2.
    """
    n = len(arr1)
    assert len(arr2) == n
    mapping = {}
    for i in range(n):
        if (arr1[i] in mapping.keys()) and (mapping[arr1[i]] != arr2[i]):
            raise Exception(f"{arr1[i]} from first array corresponds to multiple values in second array (at least {mapping[arr1[i]]} and {arr2[i]})")
        else:
            mapping[arr1[i]] = arr2[i]
    # Might as well sort the dictionary
    return dict(sorted(mapping.items()))

def translate_array_with_mapping(arr: np.ndarray, translator: dict[int, int]) -> np.ndarray:
    translated_lst = []
    for element in arr:
        translated_lst.append(translator[element])
    return np.array(translated_lst)

labels1_to_sub_domain_text = get_mapping_from_paired_arrays(original_class_labels, sub_domain_text_list)
print("Mapping of labels1 values to sub_domain_text values:", labels1_to_sub_domain_text)
class_label_binarizer = get_mapping_from_paired_arrays(original_class_labels, binarized_class_labels)
print("Mapping of labels1 values to labels2 values (our class label binarizer for domain adaptation learning):", class_label_binarizer)

labels1_to_domain_text = get_mapping_from_paired_arrays(original_class_labels, domain_text_list)
print("Mapping of labels1 values to domain_text values:", labels1_to_domain_text)

# Don't have trained DANNs yet, so do not need to define source/target domains yet. 

def print_array_size_gb(arr: np.ndarray):
    print(f"Size: length {len(arr)}, {(arr.nbytes / 1e9):.3f} GB ({(arr.nbytes / ((2**10)**3)):.3f} GiB)")

KeysView(NpzFile '/data/adenocarcinoma_dataset.npz' with keys: images, labels1, labels2, domain_text, sub_domain_text)
Unique labels1:  [ 7  8 14 15 16 17 18 19 20]
Unique labels2 (prediction target):  [0 1]
Unique domain_text:  ['Breast Cancer' 'Kidney Cancer' 'Lung and Colon Cancer']
Unique sub_domain_text:  ['breast_benign' 'breast_malignant' 'colon_aca' 'colon_bnt'
 'kidney_normal' 'kidney_tumor' 'lung_aca' 'lung_bnt' 'lung_scc']
Mapping of labels1 values to sub_domain_text values: {7: 'breast_benign', 8: 'breast_malignant', 14: 'colon_aca', 15: 'colon_bnt', 16: 'kidney_normal', 17: 'kidney_tumor', 18: 'lung_aca', 19: 'lung_bnt', 20: 'lung_scc'}
Mapping of labels1 values to labels2 values (our class label binarizer for domain adaptation learning): {7: 0, 8: 1, 14: 1, 15: 0, 16: 0, 17: 1, 18: 1, 19: 0, 20: 1}
Mapping of labels1 values to domain_text values: {7: 'Breast Cancer', 8: 'Breast Cancer', 14: 'Lung and Colon Cancer', 15: 'Lung and Colon Cancer', 16: 'Kidney Cancer', 17: 'Ki

In [ ]:
images = data['images']

In [ ]:
print_array_size_gb(images)

In [ ]:
# based on `AC_Pipeline_A_Experiments.ipynb` - assume it will give the same split based on the testing done in that notebook
TRANSFORM = cu.get_standard_transforms()
train_ds, val_ds, test_ds, source_test_ds, metadata = cu.create_domain_datasets(
    images=images,
    labels=binarized_class_labels,
    domain_text_list=domain_text_list,
    sub_domain_text_list=sub_domain_text_list,
    target_domain=TEST_DOMAIN,
    train_val_ratio=0.8,
    transform=TRANSFORM,
    branched_mode=False,
    include_source_test=True,  # Enable source domain test set
    source_test_ratio=0.1,  # 10% for source testing
    seed=42  # Use global seed
)

print_array_size_gb(train_ds.images)
print_array_size_gb(val_ds.images)
print_array_size_gb(test_ds.images)
print_array_size_gb(source_test_ds.images)

In [ ]:
example_instance = train_ds[0]
print(example_instance.keys())

example_image = example_instance['pixel_values']
example_image_device = example_image.unsqueeze(0).to(device)
print(example_image.shape)
print(example_image.mean(),example_image.std())
print(example_image.device)
print(example_image_device.shape)
print(example_image_device.device)

example_label = example_instance['labels']
print(example_label)
print(type(example_label))
print(type(example_label.item()))

example_logits = model_forward_func(example_image_device)
print(example_logits.shape)
example_prediction = torch.argmax(example_logits, dim=-1).detach().cpu().numpy()[0]
print(example_prediction)

del example_image_device